# Lesson 19 — 并发任务：采样、发送和按钮不能互相拖住 — Solution

这是独立答案 Notebook。先完成主课再打开它。

In [ ]:
from pathlib import Path
project = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tools/notebook_setup.py").is_file())
exec((project / "tools/notebook_setup.py").read_text())

In [ ]:
%%academy_lab 19-concurrent-tasks solution exercise pass
#pragma once
#include "academy/device.hpp"
namespace academy {
class Controller {
public:
    explicit Controller(Device& device) : device_(device) {}
    void tick();
private:
    Device& device_;
    std::uint32_t next_sensor_ms_ = 0;
    std::uint32_t next_network_ms_ = 0;
    bool previous_button_ = false;
    int pending_samples_ = 0;
};
}

#include <string>
namespace academy {
void Controller::tick() {
    const auto now = device_.input.now_ms;
    if (now >= next_sensor_ms_) {
        device_.output.reading = device_.input.temperature;
        ++pending_samples_;
        next_sensor_ms_ = now + 100;
    }
    if (device_.input.connected && pending_samples_ > 0 && now >= next_network_ms_) {
        --pending_samples_;
        ++device_.output.events;
        device_.output.outgoing = "temperature sample";
        next_network_ms_ = now + 50;
    }
    if (device_.input.button && !previous_button_) device_.output.led = !device_.output.led;
    previous_button_ = device_.input.button;
    device_.output.queue_depth = pending_samples_;
    device_.output.state = device_.input.connected ? "online" : "offline";
}
}
